# Лабораторная работа №4: Кластеризация текстов Amazon Fine Food Reviews

**Цель:** Рассмотреть прикладные задачи обработки естественного языка и изучить инструменты для работы с готовыми моделями.

**Задачи:**
1. Загрузить и предобработать датасет Amazon Fine Food Reviews
2. Выполнить кодирование текстов методами TF-IDF и SBERT
3. Провести кластеризацию с использованием K-Means
4. Оценить качество кластеризации метриками
5. Извлечь топ-слова для каждого кластера
6. Построить облака слов
7. Проинтерпретировать результаты

## 1. Импорт библиотек и настройка окружения

In [16]:
# Базовые библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Предобработка текста
import re
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Векторизация и кластеризация
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer

# Извлечение ключевых слов и облака слов
from keybert import KeyBERT
from wordcloud import WordCloud

# Прогресс-бар
from tqdm import tqdm
tqdm.pandas()

# Настройка визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Все библиотеки успешно импортированы!")

Все библиотеки успешно импортированы!


In [17]:
# Скачивание необходимых ресурсов NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("NLTK ресурсы загружены!")

NLTK ресурсы загружены!


## 2. Загрузка датасета Amazon Fine Food Reviews

Датасет содержит отзывы на продукты питания с Amazon. Мы возьмем часть датасета для оптимальной скорости обработки.

Используем `kagglehub` для автоматической загрузки датасета.

In [ ]:
# Загрузка датасета из локального файла Reviews.csv
# Скачайте файл с Kaggle: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
# И поместите его в директорию с notebook

print("Загрузка датасета Amazon Fine Food Reviews...")

# Загрузка CSV файла с указанием кодировки
df_sample = pd.DataFrame(pd.read_csv(
    'Reviews.csv',
    nrows=15000,
)['Text'])

print(f"Загружено отзывов: {len(df_sample)}")
print(f"\\nСтолбцы датасета: {list(df_sample.columns)}")
print(f"\\nПервые строки датасета:")
df_sample.head()

In [ ]:
# Информация о датасете
print("Информация о датасете:")
df_sample.info()

print("\\nПервые 5 отзывов:")
print(df_sample['Text'].head())

In [20]:
# Проверка на пропуски и удаление строк с пустыми текстами
print(f"Пропущенные значения в 'Text': {df_sample['Text'].isna().sum()}")
df_sample = df_sample.dropna(subset=['Text'])
df_sample = df_sample[df_sample['Text'].str.strip() != '']
print(f"Отзывов после очистки: {len(df_sample)}")

Пропущенные значения в 'Text': 0
Отзывов после очистки: 15000


## 3. Предобработка текстов

Выполним:
- Удаление HTML-тегов
- Удаление смайликов и эмодзи
- Удаление знаков препинания и специальных символов
- Приведение к нижнему регистру
- Токенизация
- Удаление стоп-слов
- Лемматизация

In [21]:
# Инициализация инструментов для предобработки
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Полная предобработка текста:
    - Удаление HTML-тегов
    - Удаление эмодзи и смайликов
    - Приведение к нижнему регистру
    - Удаление пунктуации и цифр
    - Токенизация
    - Удаление стоп-слов
    - Лемматизация
    """
    # Удаление HTML-тегов
    text = BeautifulSoup(text, 'html.parser').get_text()
    
    # Удаление эмодзи и смайликов
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    # Приведение к нижнему регистру
    text = text.lower()
    
    # Удаление знаков препинания и цифр, оставляем только буквы
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Удаление множественных пробелов
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Токенизация
    tokens = word_tokenize(text)
    
    # Удаление стоп-слов и коротких слов (менее 3 символов)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    
    # Лемматизация
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

print("Функция предобработки создана!")

Функция предобработки создана!


In [22]:
# Пример работы функции предобработки
sample_text = df_sample['Text'].iloc[0]
print("Исходный текст:")
print(sample_text)
print("\n" + "="*80 + "\n")
print("Обработанный текст:")
print(preprocess_text(sample_text))

Исходный текст:
Having tried a couple of other brands of gluten-free sandwich cookies, these are the best of the bunch.  They're crunchy and true to the texture of the other "real" cookies that aren't gluten-free.  Some might think that the filling makes them a bit too sweet, but for me that just means I've satisfied my sweet tooth sooner!  The chocolate version from Glutino is just as good and has a true "chocolatey" taste - something that isn't there with the other gluten-free brands out there.


Обработанный текст:
tried couple brand gluten free sandwich cooky best bunch crunchy true texture real cooky gluten free might think filling make bit sweet mean satisfied sweet tooth sooner chocolate version glutino good true chocolatey taste something gluten free brand


In [23]:
# Применение предобработки ко всем текстам
print("Предобработка текстов (это может занять несколько минут)...")
df_sample['processed_text'] = df_sample['Text'].progress_apply(preprocess_text)

# Удаление пустых текстов после обработки
df_sample = df_sample[df_sample['processed_text'].str.strip() != '']
print(f"\nОтзывов после предобработки: {len(df_sample)}")

Предобработка текстов (это может занять несколько минут)...


100%|██████████| 15000/15000 [00:06<00:00, 2155.07it/s]


Отзывов после предобработки: 15000


In [ ]:
# Сохраним обработанные данные
texts_original = df_sample['Text'].tolist()
texts_processed = df_sample['processed_text'].tolist()

print(f"Всего текстов для анализа: {len(texts_processed)}")
print(f"\\nПримеры обработанных текстов:")
for i in range(min(3, len(texts_processed))):
    print(f"\\n{i+1}. {texts_processed[i][:200]}...")

## 4. Векторизация текстов методом TF-IDF

In [25]:
# Создание TF-IDF векторизатора
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,      # Максимум 5000 слов в словаре
    max_df=0.8,             # Игнорируем слова, которые встречаются в >80% документов
    min_df=5,               # Игнорируем слова, которые встречаются в <5 документах
    ngram_range=(1, 2)      # Используем униграммы и биграммы
)

# Векторизация обработанных текстов
print("Векторизация текстов с помощью TF-IDF...")
tfidf_matrix = tfidf_vectorizer.fit_transform(texts_processed)

print(f"Размер TF-IDF матрицы: {tfidf_matrix.shape}")
print(f"Количество документов: {tfidf_matrix.shape[0]}")
print(f"Размер словаря (количество признаков): {tfidf_matrix.shape[1]}")

Векторизация текстов с помощью TF-IDF...
Размер TF-IDF матрицы: (15000, 5000)
Количество документов: 15000
Размер словаря (количество признаков): 5000


In [26]:
# Получение словаря TF-IDF
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"Первые 20 слов из словаря TF-IDF: {list(tfidf_feature_names[:20])}")

Первые 20 слов из словаря TF-IDF: ['ability', 'able', 'able buy', 'able find', 'able get', 'absolute', 'absolute favorite', 'absolutely', 'absolutely best', 'absolutely delicious', 'absolutely love', 'absorb', 'acai', 'accept', 'acceptable', 'accident', 'according', 'account', 'accurate', 'acid']


## 5. Векторизация текстов с помощью SBERT (Sentence-BERT)

In [ ]:
# Загрузка SBERT модели
print("Загрузка SBERT модели...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Модель загружена!")

# Кодирование текстов (используем оригинальные тексты, а не обработанные)
print("\nКодирование текстов с помощью SBERT (это может занять несколько минут)...")
sbert_embeddings = sbert_model.encode(
    texts_original,
    show_progress_bar=True,
    batch_size=32
)

print(f"\nРазмер SBERT эмбеддингов: {sbert_embeddings.shape}")
print(f"Количество документов: {sbert_embeddings.shape[0]}")
print(f"Размерность эмбеддинга: {sbert_embeddings.shape[1]}")

Загрузка SBERT модели...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

## 6. Определение оптимального количества кластеров

Используем метод локтя (Elbow Method) и Silhouette Score

In [ ]:
# Функция для определения оптимального количества кластеров
def find_optimal_clusters(data, max_k=10, sample_size=5000):
    """
    Определяет оптимальное количество кластеров используя метод локтя и Silhouette Score
    """
    # Для ускорения используем подвыборку
    if len(data) > sample_size:
        indices = np.random.choice(len(data), sample_size, replace=False)
        data_sample = data[indices]
    else:
        data_sample = data
    
    inertias = []
    silhouette_scores = []
    k_range = range(2, max_k + 1)
    
    print("Поиск оптимального количества кластеров...")
    for k in tqdm(k_range):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(data_sample)
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(data_sample, kmeans.labels_))
    
    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Метод локтя
    axes[0].plot(k_range, inertias, 'bo-')
    axes[0].set_xlabel('Количество кластеров (k)')
    axes[0].set_ylabel('Inertia')
    axes[0].set_title('Метод локтя')
    axes[0].grid(True)
    
    # Silhouette Score
    axes[1].plot(k_range, silhouette_scores, 'ro-')
    axes[1].set_xlabel('Количество кластеров (k)')
    axes[1].set_ylabel('Silhouette Score')
    axes[1].set_title('Silhouette Score')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Определяем оптимальное k по максимальному Silhouette Score
    optimal_k = k_range[np.argmax(silhouette_scores)]
    print(f"\nОптимальное количество кластеров: {optimal_k}")
    return optimal_k

In [ ]:
# Определение оптимального количества кластеров для TF-IDF
print("=" * 80)
print("Анализ для TF-IDF эмбеддингов")
print("=" * 80)
optimal_k_tfidf = find_optimal_clusters(tfidf_matrix.toarray(), max_k=10)

In [ ]:
# Определение оптимального количества кластеров для SBERT
print("=" * 80)
print("Анализ для SBERT эмбеддингов")
print("=" * 80)
optimal_k_sbert = find_optimal_clusters(sbert_embeddings, max_k=10)

In [ ]:
# Используем одинаковое количество кластеров для обоих методов (для честного сравнения)
# Выбираем среднее или максимальное
N_CLUSTERS = max(optimal_k_tfidf, optimal_k_sbert)
print(f"\nИтоговое количество кластеров для обоих методов: {N_CLUSTERS}")

## 7. Кластеризация с использованием K-Means

In [ ]:
# Кластеризация TF-IDF эмбеддингов
print("Кластеризация TF-IDF эмбеддингов...")
kmeans_tfidf = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10, max_iter=300)
labels_tfidf = kmeans_tfidf.fit_predict(tfidf_matrix)

print(f"Кластеризация завершена!")
print(f"Распределение по кластерам:")
unique, counts = np.unique(labels_tfidf, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"  Кластер {cluster}: {count} документов")

In [ ]:
# Кластеризация SBERT эмбеддингов
print("Кластеризация SBERT эмбеддингов...")
kmeans_sbert = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10, max_iter=300)
labels_sbert = kmeans_sbert.fit_predict(sbert_embeddings)

print(f"Кластеризация завершена!")
print(f"Распределение по кластерам:")
unique, counts = np.unique(labels_sbert, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"  Кластер {cluster}: {count} документов")

## 8. Визуализация кластеров с помощью t-SNE

In [ ]:
# Функция для визуализации кластеров
def visualize_clusters(embeddings, labels, title, sample_size=3000):
    """
    Визуализация кластеров с использованием t-SNE
    """
    # Используем подвыборку для ускорения t-SNE
    if len(embeddings) > sample_size:
        indices = np.random.choice(len(embeddings), sample_size, replace=False)
        embeddings_sample = embeddings[indices]
        labels_sample = labels[indices]
    else:
        embeddings_sample = embeddings
        labels_sample = labels
    
    print(f"Применение t-SNE для {title}...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
    embeddings_2d = tsne.fit_transform(embeddings_sample)
    
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(
        embeddings_2d[:, 0],
        embeddings_2d[:, 1],
        c=labels_sample,
        cmap='tab10',
        alpha=0.6,
        s=50
    )
    plt.colorbar(scatter, label='Кластер')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('t-SNE компонента 1')
    plt.ylabel('t-SNE компонента 2')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Визуализация TF-IDF кластеров
visualize_clusters(
    tfidf_matrix.toarray(),
    labels_tfidf,
    'Визуализация кластеров TF-IDF (t-SNE)'
)

In [ ]:
# Визуализация SBERT кластеров
visualize_clusters(
    sbert_embeddings,
    labels_sbert,
    'Визуализация кластеров SBERT (t-SNE)'
)

## 9. Оценка качества кластеризации

In [ ]:
# Функция для вычисления метрик
def calculate_clustering_metrics(embeddings, labels, method_name):
    """
    Вычисление метрик качества кластеризации:
    - Silhouette Score: [-1, 1], выше лучше
    - Davies-Bouldin Index: [0, inf), ниже лучше
    - Calinski-Harabasz Index: [0, inf), выше лучше
    """
    silhouette = silhouette_score(embeddings, labels)
    davies_bouldin = davies_bouldin_score(embeddings, labels)
    calinski_harabasz = calinski_harabasz_score(embeddings, labels)
    
    print(f"\n{'='*80}")
    print(f"Метрики качества кластеризации для {method_name}")
    print(f"{'='*80}")
    print(f"Silhouette Score:          {silhouette:.4f}  (выше - лучше, диапазон [-1, 1])")
    print(f"Davies-Bouldin Index:      {davies_bouldin:.4f}  (ниже - лучше)")
    print(f"Calinski-Harabasz Index:   {calinski_harabasz:.4f}  (выше - лучше)")
    print(f"{'='*80}")
    
    return {
        'silhouette': silhouette,
        'davies_bouldin': davies_bouldin,
        'calinski_harabasz': calinski_harabasz
    }

In [ ]:
# Метрики для TF-IDF
metrics_tfidf = calculate_clustering_metrics(
    tfidf_matrix.toarray(),
    labels_tfidf,
    'TF-IDF'
)

In [ ]:
# Метрики для SBERT
metrics_sbert = calculate_clustering_metrics(
    sbert_embeddings,
    labels_sbert,
    'SBERT'
)

In [ ]:
# Сравнительная таблица метрик
comparison_df = pd.DataFrame({
    'Метрика': ['Silhouette Score', 'Davies-Bouldin Index', 'Calinski-Harabasz Index'],
    'TF-IDF': [
        metrics_tfidf['silhouette'],
        metrics_tfidf['davies_bouldin'],
        metrics_tfidf['calinski_harabasz']
    ],
    'SBERT': [
        metrics_sbert['silhouette'],
        metrics_sbert['davies_bouldin'],
        metrics_sbert['calinski_harabasz']
    ]
})

print("\nСравнительная таблица метрик:")
print(comparison_df.to_string(index=False))

## 10. Извлечение топ-слов для кластеров TF-IDF

In [ ]:
# Функция для извлечения топ-слов из TF-IDF кластеров
def get_top_words_tfidf(tfidf_matrix, labels, feature_names, n_clusters, top_n=15):
    """
    Извлечение топ-слов для каждого кластера на основе TF-IDF весов
    """
    cluster_keywords = {}
    
    for cluster_id in range(n_clusters):
        # Индексы документов в кластере
        cluster_indices = np.where(labels == cluster_id)[0]
        
        # Усреднение TF-IDF векторов документов кластера
        cluster_center = tfidf_matrix[cluster_indices].mean(axis=0).A1
        
        # Получение индексов топ-слов
        top_indices = cluster_center.argsort()[-top_n:][::-1]
        
        # Получение слов и их весов
        top_words = [(feature_names[i], cluster_center[i]) for i in top_indices]
        cluster_keywords[cluster_id] = top_words
    
    return cluster_keywords

In [ ]:
# Извлечение топ-слов для TF-IDF кластеров
print("Извлечение топ-слов для TF-IDF кластеров...\n")
tfidf_keywords = get_top_words_tfidf(
    tfidf_matrix,
    labels_tfidf,
    tfidf_feature_names,
    N_CLUSTERS,
    top_n=15
)

# Вывод топ-слов
for cluster_id, keywords in tfidf_keywords.items():
    print(f"\n{'='*80}")
    print(f"Кластер {cluster_id} (TF-IDF) - Топ-15 слов:")
    print(f"{'='*80}")
    for word, weight in keywords:
        print(f"  {word:20s} : {weight:.4f}")

## 11. Извлечение топ-слов для кластеров SBERT с помощью KeyBERT

In [ ]:
# Инициализация KeyBERT
kw_model = KeyBERT(model=sbert_model)
print("KeyBERT модель инициализирована!")

In [ ]:
# Функция для извлечения топ-слов с помощью KeyBERT
def get_top_words_keybert(texts, labels, n_clusters, kw_model, top_n=15):
    """
    Извлечение ключевых слов для каждого кластера с помощью KeyBERT
    """
    cluster_keywords = {}
    
    for cluster_id in range(n_clusters):
        # Индексы документов в кластере
        cluster_indices = np.where(labels == cluster_id)[0]
        
        # Объединяем все тексты кластера
        cluster_text = ' '.join([texts[i] for i in cluster_indices])
        
        # Извлекаем ключевые слова
        keywords = kw_model.extract_keywords(
            cluster_text,
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=top_n,
            use_maxsum=True,
            nr_candidates=20,
            diversity=0.5
        )
        
        cluster_keywords[cluster_id] = keywords
    
    return cluster_keywords

In [ ]:
# Извлечение топ-слов для SBERT кластеров
print("Извлечение топ-слов для SBERT кластеров с помощью KeyBERT...\n")
sbert_keywords = get_top_words_keybert(
    texts_processed,
    labels_sbert,
    N_CLUSTERS,
    kw_model,
    top_n=15
)

# Вывод топ-слов
for cluster_id, keywords in sbert_keywords.items():
    print(f"\n{'='*80}")
    print(f"Кластер {cluster_id} (SBERT + KeyBERT) - Топ-15 ключевых слов:")
    print(f"{'='*80}")
    for word, score in keywords:
        print(f"  {word:30s} : {score:.4f}")

## 12. Построение облаков слов для кластеров TF-IDF

In [ ]:
# Функция для построения облака слов
def create_wordcloud(texts, labels, cluster_id, title):
    """
    Создание облака слов для конкретного кластера
    """
    # Получаем тексты кластера
    cluster_indices = np.where(labels == cluster_id)[0]
    cluster_text = ' '.join([texts[i] for i in cluster_indices])
    
    # Создаем облако слов
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='viridis',
        max_words=100,
        relative_scaling=0.5,
        min_font_size=10
    ).generate(cluster_text)
    
    # Визуализация
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Построение облаков слов для каждого TF-IDF кластера
print("Построение облаков слов для TF-IDF кластеров...\n")
for cluster_id in range(N_CLUSTERS):
    create_wordcloud(
        texts_processed,
        labels_tfidf,
        cluster_id,
        f'Облако слов для кластера {cluster_id} (TF-IDF)'
    )

## 13. Построение облаков слов для кластеров SBERT

In [ ]:
# Построение облаков слов для каждого SBERT кластера
print("Построение облаков слов для SBERT кластеров...\n")
for cluster_id in range(N_CLUSTERS):
    create_wordcloud(
        texts_processed,
        labels_sbert,
        cluster_id,
        f'Облако слов для кластера {cluster_id} (SBERT)'
    )

## 14. Примеры текстов из каждого кластера

In [ ]:
# Функция для вывода примеров из кластеров
def show_cluster_examples(texts, labels, n_clusters, n_examples=3, method_name=''):
    """
    Вывод примеров текстов из каждого кластера
    """
    print(f"\n{'='*80}")
    print(f"Примеры текстов из кластеров ({method_name})")
    print(f"{'='*80}\n")
    
    for cluster_id in range(n_clusters):
        cluster_indices = np.where(labels == cluster_id)[0]
        
        print(f"\n{'-'*80}")
        print(f"Кластер {cluster_id} ({len(cluster_indices)} документов)")
        print(f"{'-'*80}")
        
        # Выбираем случайные примеры
        sample_indices = np.random.choice(
            cluster_indices,
            min(n_examples, len(cluster_indices)),
            replace=False
        )
        
        for i, idx in enumerate(sample_indices, 1):
            print(f"\nПример {i}:")
            print(f"{texts[idx][:300]}...")

In [ ]:
# Примеры для TF-IDF кластеров
show_cluster_examples(texts_original, labels_tfidf, N_CLUSTERS, n_examples=3, method_name='TF-IDF')

In [ ]:
# Примеры для SBERT кластеров
show_cluster_examples(texts_original, labels_sbert, N_CLUSTERS, n_examples=3, method_name='SBERT')

## 15. Интерпретация кластеров и выводы

In [ ]:
# Создаем функцию для интерпретации кластеров
def interpret_cluster(cluster_id, keywords_list, method_name):
    """
    Вспомогательная функция для интерпретации кластера
    """
    print(f"\n{'='*80}")
    print(f"Интерпретация кластера {cluster_id} ({method_name})")
    print(f"{'='*80}")
    print(f"\nКлючевые слова: {', '.join([word for word, _ in keywords_list[:10]])}")
    print(f"\nВозможная тематика кластера:")
    print("(Здесь должна быть ваша интерпретация на основе ключевых слов и примеров текстов)")

print("\n" + "#" * 80)
print("# ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ КЛАСТЕРИЗАЦИИ")
print("#" * 80)

### Интерпретация кластеров TF-IDF

На основе топ-слов, облаков слов и примеров текстов проинтерпретируйте каждый кластер:
- Какая основная тематика?
- О каких продуктах идет речь?
- Позитивные или негативные отзывы?
- Особенности языка и стиля

In [ ]:
# Интерпретация TF-IDF кластеров
for cluster_id in range(N_CLUSTERS):
    interpret_cluster(cluster_id, tfidf_keywords[cluster_id], 'TF-IDF')

### Интерпретация кластеров SBERT

Проведите аналогичную интерпретацию для кластеров SBERT

In [ ]:
# Интерпретация SBERT кластеров
for cluster_id in range(N_CLUSTERS):
    interpret_cluster(cluster_id, sbert_keywords[cluster_id], 'SBERT')

### Общие выводы

**Сравнение методов TF-IDF и SBERT:**

1. **Качество кластеризации (по метрикам):**
   - Silhouette Score: ...
   - Davies-Bouldin Index: ...
   - Calinski-Harabasz Index: ...

2. **Интерпретируемость результатов:**
   - TF-IDF: ...
   - SBERT: ...

3. **Семантическая осмысленность кластеров:**
   - TF-IDF: ...
   - SBERT: ...

4. **Преимущества и недостатки каждого подхода:**
   - TF-IDF:
     - Преимущества: ...
     - Недостатки: ...
   - SBERT:
     - Преимущества: ...
     - Недостатки: ...

5. **Рекомендации по использованию:**
   - Когда лучше использовать TF-IDF: ...
   - Когда лучше использовать SBERT: ...

In [ ]:
print("\n" + "#" * 80)
print("# РАБОТА ЗАВЕРШЕНА")
print("#" * 80)
print("\nВсе этапы лабораторной работы успешно выполнены:")
print("✓ Корпус документов предобработан и токенизирован")
print("✓ Тексты векторизированы двумя способами (TF-IDF и SBERT)")
print("✓ Проведена кластеризация с использованием K-Means")
print("✓ Вычислены метрики качества кластеризации")
print("✓ Извлечены топ-слова для каждого подхода")
print("✓ Построены облака слов для всех кластеров")
print("✓ Выполнена интерпретация результатов")